# Browse all cross-elicit responses

Unified browser for all three run types in `cross-elicit/results/`:

| `run_type` | score file | row key | col key | response source |
|---|---|---|---|---|
| `'finetuned'` | `scores_<model>.json` | pole (e.g. `agreeableness-plus`) | eval (e.g. `agreeableness`) | `eval_results/finetuning/<dirname>/rows.jsonl` |
| `'sysprompts'` | `scores_sysprompts_<model>.json` | pole (e.g. `agreeableness--agreeable`) | eval (e.g. `agreeableness`) | `eval_results/sys_prompts/<dirname>/rows.jsonl` |
| `'orthogonality'` | `eval-orthogonality_scores.json` | row_label (e.g. `agreeableness_score`) | col_label (e.g. `agreeableness.agreeable`) | `eval_results/test_evals/.../judgments.jsonl` |

**Finetuned / sysprompts** — use `get_responses(model, pole, eval, run_type=...)` and `get_scores(...)`.

**Orthogonality** — use `get_orthogonality_responses(row_label, col_label)` and `get_orthogonality_scores(...)`.

Run the setup and helpers cells first, then use the listing cell to see what's available.

In [ ]:
import json
from pathlib import Path

_here = Path('.').resolve()
RESULTS_DIR = _here if _here.name == 'results' else _here / 'results'
EVAL_ROOT = (RESULTS_DIR.parent / 'eval_results').resolve()

EVAL_ROOTS = {
    'finetuned':  EVAL_ROOT / 'finetuning',
    'sysprompts': EVAL_ROOT / 'sys_prompts',
}

# SCORES['finetuned'][model]  = doc  (scores_<model>.json)
# SCORES['sysprompts'][model] = doc  (scores_sysprompts_<model>.json)
# SCORES['orthogonality']     = doc  (eval-orthogonality_scores.json)
SCORES = {'finetuned': {}, 'sysprompts': {}}

for p in sorted(RESULTS_DIR.glob('scores_*.json')):
    if 'sysprompts' in p.stem:
        continue
    doc = json.loads(p.read_text())
    if doc.get('n_cells', 0) > 0:
        SCORES['finetuned'][doc['base_model']] = doc

for p in sorted(RESULTS_DIR.glob('scores_sysprompts_*.json')):
    doc = json.loads(p.read_text())
    if doc.get('n_cells', 0) > 0:
        SCORES['sysprompts'][doc['base_model']] = doc

orth_path = RESULTS_DIR / 'eval-orthogonality_scores.json'
if orth_path.exists():
    doc = json.loads(orth_path.read_text())
    if doc.get('n_cells', 0) > 0:
        SCORES['orthogonality'] = doc

print('Loaded:')
for model, doc in SCORES['finetuned'].items():
    print(f"  finetuned    {model}  ({doc['n_cells']} cells, {doc['n_poles']} poles)")
for model, doc in SCORES['sysprompts'].items():
    print(f"  sysprompts   {model}  ({doc['n_cells']} cells, {doc['n_poles']} poles)")
if 'orthogonality' in SCORES:
    d = SCORES['orthogonality']
    print(f"  orthogonality  {d['n_rows']} rows × {d['n_cols']} cols  ({d['n_cells']} cells)")
print()
print("Use get_responses(model, pole, eval, run_type='finetuned'/'sysprompts')")
print("Use get_orthogonality_responses(row_label, col_label)")

In [ ]:
# ── finetuned / sysprompts ────────────────────────────────────────────────

def _iter_rows(model, pole, eval_propensity, run_type):
    if run_type not in ('finetuned', 'sysprompts'):
        raise ValueError(f"run_type must be 'finetuned' or 'sysprompts'; got {run_type!r}")
    bucket = SCORES[run_type]
    if model not in bucket:
        raise KeyError(f"unknown model {model!r} for {run_type}; available: {sorted(bucket)}")
    cells = bucket[model]['cells']
    if pole not in cells:
        raise KeyError(
            f"unknown pole {pole!r} for {run_type}/{model}; "
            f"available: {sorted(cells)}"
        )
    if eval_propensity not in cells[pole]:
        raise KeyError(
            f"no eval {eval_propensity!r} for {run_type}/{model}/{pole}; "
            f"available: {sorted(cells[pole])}"
        )
    rows_path = EVAL_ROOTS[run_type] / cells[pole][eval_propensity]['meta']['dirname'] / 'rows.jsonl'
    if not rows_path.exists():
        raise FileNotFoundError(f'missing rows.jsonl: {rows_path}')
    with rows_path.open() as f:
        for line in f:
            yield json.loads(line)


def get_responses(model, pole, eval_propensity, run_type='finetuned'):
    """Conversations from a finetuned checkpoint or sys-prompted run.

    Args:
        model: base LLM name, e.g. 'meta-llama-Llama-3.1-8B-Instruct'
        pole:  finetuning pole or sys-prompt identity.
               finetuned  → e.g. 'agreeableness-plus', 'base'
               sysprompts → e.g. 'agreeableness--agreeable', 'baseline-empty'
        eval_propensity: propensity being judged, e.g. 'agreeableness'
        run_type: 'finetuned' (default) or 'sysprompts'

    Returns list of {'question', 'answer'} dicts.
    """
    return [
        {'question': r.get('question'), 'answer': r.get('answer')}
        for r in _iter_rows(model, pole, eval_propensity, run_type)
    ]


def get_scores(model, pole, eval_propensity, run_type='finetuned'):
    """Per-conversation judge scores, in the same order as get_responses(...).

    Entries are int/float; None when the judge returned null/fail.
    """
    return [r.get('score') for r in _iter_rows(model, pole, eval_propensity, run_type)]


# ── orthogonality ─────────────────────────────────────────────────────────

def _iter_orthogonality_rows(row_label, col_label):
    if 'orthogonality' not in SCORES:
        raise RuntimeError('eval-orthogonality_scores.json not loaded')
    doc = SCORES['orthogonality']
    if row_label not in doc['cells']:
        raise KeyError(
            f"unknown row_label {row_label!r}; "
            f"available: {doc['row_labels']}"
        )
    if col_label not in doc['cells'][row_label]:
        raise KeyError(
            f"unknown col_label {col_label!r} for row {row_label!r}; "
            f"available: {sorted(doc['cells'][row_label])}"
        )
    judgments_path = Path(doc['run_dir']) / 'judgments.jsonl'
    if not judgments_path.exists():
        raise FileNotFoundError(f'missing judgments.jsonl: {judgments_path}')
    with judgments_path.open() as f:
        for line in f:
            r = json.loads(line)
            if r.get('judge_key') == row_label and r.get('col_label') == col_label:
                yield r


def get_orthogonality_responses(row_label, col_label):
    """Conversations from the orthogonality run for a given (row_label, col_label) cell.

    Args:
        row_label: judge-prompt key, e.g. 'agreeableness_score'
                   (see SCORES['orthogonality']['row_labels'] for all values)
        col_label: response-set key, e.g. 'agreeableness.agreeable'
                   (see SCORES['orthogonality']['col_labels'] for all values)

    Returns list of {'question', 'answer'} dicts.
    """
    return [
        {'question': r.get('question'), 'answer': r.get('answer')}
        for r in _iter_orthogonality_rows(row_label, col_label)
    ]


def get_orthogonality_scores(row_label, col_label):
    """Per-conversation judge scores for an orthogonality cell.

    Entries are int/float; None when the judge returned null/fail.
    """
    return [r.get('score') for r in _iter_orthogonality_rows(row_label, col_label)]

In [ ]:
# Run this cell to list what poles and evals are available per model / run type.

for run_type in ('finetuned', 'sysprompts'):
    for model, doc in SCORES[run_type].items():
        cells = doc['cells']
        print(f'[{run_type}] {model}')
        for pole, evals in sorted(cells.items()):
            print(f'  pole={pole!r:50s}  evals={sorted(evals)}')
        print()

if 'orthogonality' in SCORES:
    doc = SCORES['orthogonality']
    print('[orthogonality]')
    print('  row_labels:', doc['row_labels'])
    print('  col_labels:', doc['col_labels'])

## Examples

In [ ]:
# ── finetuned example ────────────────────────────────────────────────────
if SCORES['finetuned']:
    model = next(iter(SCORES['finetuned']))
    cells = SCORES['finetuned'][model]['cells']
    pole  = 'base' if 'base' in cells else next(iter(cells))
    eval_p = next(iter(cells[pole]))

    convos = get_responses(model, pole, eval_p, run_type='finetuned')
    scores = get_scores(model, pole, eval_p, run_type='finetuned')
    print(f'[finetuned] {model} | {pole} | {eval_p}: {len(convos)} convos')
    print()
    print('Q:', (convos[0]['question'] or '')[:300])
    print()
    print('A:', (convos[0]['answer'] or '')[:300])
    print()
    print('score:', scores[0])

In [ ]:
# ── sysprompts example ───────────────────────────────────────────────────
if SCORES['sysprompts']:
    model = next(iter(SCORES['sysprompts']))
    cells = SCORES['sysprompts'][model]['cells']
    pole  = 'baseline-empty' if 'baseline-empty' in cells else next(iter(cells))
    eval_p = next(iter(cells[pole]))

    convos = get_responses(model, pole, eval_p, run_type='sysprompts')
    scores = get_scores(model, pole, eval_p, run_type='sysprompts')
    print(f'[sysprompts] {model} | {pole} | {eval_p}: {len(convos)} convos')
    print()
    print('Q:', (convos[0]['question'] or '')[:300])
    print()
    print('A:', (convos[0]['answer'] or '')[:300])
    print()
    print('score:', scores[0])

In [ ]:
# ── orthogonality example ────────────────────────────────────────────────
if 'orthogonality' in SCORES:
    doc   = SCORES['orthogonality']
    row_l = doc['row_labels'][0]
    col_l = doc['col_labels'][0]

    convos = get_orthogonality_responses(row_l, col_l)
    scores = get_orthogonality_scores(row_l, col_l)
    print(f'[orthogonality] row={row_l!r} | col={col_l!r}: {len(convos)} convos')
    print()
    print('Q:', (convos[0]['question'] or '')[:300])
    print()
    print('A:', (convos[0]['answer'] or '')[:300])
    print()
    print('score:', scores[0])